# 1. Azure Policy & Key Vault — Implementation

Two of the most exam-heavy topics in AZ-500 Domain 4 live here:

* **Azure Policy** — guardrails that *audit*, *deny*, or *auto-fix* resource configuration.
* **Azure Key Vault** — the place where secrets, keys and certificates should live (and often don't).

### What you'll build in this notebook

1. A tiny **Python policy engine** that evaluates the same JSON shape Azure Policy uses.
2. A **compliance dashboard** for an initiative (a bundle of policies).
3. A **Key Vault decision helper**: RBAC vs legacy access policy, soft-delete / purge protection lifecycle, and a rotation-due scanner.
4. A **bad → best** hardening walkthrough for a production Key Vault.

Everything is simulated — no Azure subscription needed.

## Before you run this notebook

1. From the lab folder run `uv sync`.
2. In VS Code, pick the `.venv` kernel (top-right kernel picker).
3. If the kernel doesn't appear, reload the window (`Cmd+Shift+P` → *Developer: Reload Window*).

Everything runs in plain Python on the standard library.

## 1. Azure Policy — the mental model

```
Policy definition  →  describes WHAT to check + WHAT to do
Policy initiative  →  bundle of related definitions (e.g. "CIS Benchmark")
Policy assignment  →  binds a definition/initiative to a SCOPE (MG / Sub / RG)
```

### Effects you must know for the exam

| Effect                 | What happens                                         | Typical use             |
|------------------------|------------------------------------------------------|-------------------------|
| **Audit**              | Resource is created, non-compliance is *logged*      | Monitoring / reporting  |
| **Deny**               | Resource creation is **blocked**                     | Hard enforcement        |
| **Modify**             | Resource is auto-patched (e.g. add a tag)            | Auto-remediation        |
| **DeployIfNotExists**  | Deploy a related resource if missing (diagnostics)   | Auto-provisioning       |
| **AuditIfNotExists**   | Check if related resource exists                     | Monitoring              |
| **Disabled**           | Policy is not evaluated                              | Temporarily turn off    |

**Exam trap**: `Modify` and `DeployIfNotExists` need a **managed identity** on the assignment and a **remediation task** to fix existing (already-created) resources. They only fix *new* resources on their own.

In [1]:
# A minimal Azure-Policy-like evaluator.
# The shape of `policy_rule` matches the real policyRule JSON Azure accepts,
# just with fewer operators. If you can read this, you can read the real ones.

def get_field(resource, field):
    """Look up a dotted or aliased field on a resource dict."""
    # Real Azure Policy uses "aliases" like "Microsoft.Storage/storageAccounts/supportsHttpsTrafficOnly".
    # For the demo we store those keys directly on the resource.
    return resource.get(field)


def evaluate_condition(resource, cond):
    if "allOf" in cond:
        return all(evaluate_condition(resource, c) for c in cond["allOf"])
    if "anyOf" in cond:
        return any(evaluate_condition(resource, c) for c in cond["anyOf"])
    if "not" in cond:
        return not evaluate_condition(resource, cond["not"])

    value = get_field(resource, cond["field"])
    if "equals" in cond:     return value == cond["equals"]
    if "notEquals" in cond:  return value != cond["notEquals"]
    if "exists" in cond:     return (value is not None) == cond["exists"]
    if "in" in cond:         return value in cond["in"]
    if "notIn" in cond:      return value not in cond["notIn"]
    raise ValueError(f"Unknown operator in {cond}")


def evaluate_policy(resource, policy):
    rule = policy["properties"]["policyRule"]
    matched = evaluate_condition(resource, rule["if"])
    effect = rule["then"]["effect"] if matched else "compliant"
    return effect


# --- Policy: storage accounts must use HTTPS only ---
require_https = {
    "properties": {
        "displayName": "Storage accounts must use HTTPS",
        "policyRule": {
            "if": {
                "allOf": [
                    {"field": "type", "equals": "Microsoft.Storage/storageAccounts"},
                    {"field": "supportsHttpsTrafficOnly", "notEquals": True},
                ]
            },
            "then": {"effect": "deny"},
        },
    }
}

resources = [
    {"name": "sa-good",     "type": "Microsoft.Storage/storageAccounts", "supportsHttpsTrafficOnly": True},
    {"name": "sa-bad",      "type": "Microsoft.Storage/storageAccounts", "supportsHttpsTrafficOnly": False},
    {"name": "vm-01",       "type": "Microsoft.Compute/virtualMachines"},
]

print("=== Policy: 'Storage accounts must use HTTPS' ===")
for r in resources:
    effect = evaluate_policy(r, require_https)
    icon = {"deny": "⛔", "audit": "⚠️", "compliant": "✅"}.get(effect, "❔")
    print(f"  {icon} {r['name']:10s} ({r['type']}) → {effect}")


=== Policy: 'Storage accounts must use HTTPS' ===
  ✅ sa-good    (Microsoft.Storage/storageAccounts) → compliant
  ⛔ sa-bad     (Microsoft.Storage/storageAccounts) → deny
  ✅ vm-01      (Microsoft.Compute/virtualMachines) → compliant


### Reading that back

* The `sa-good` storage account matches the `type` branch but not the `notEquals True` branch, so the overall `allOf` is false → **compliant**.
* The `sa-bad` account matches both branches → effect **deny**.
* The VM doesn't match the first branch at all → **compliant** (the policy is scoped to storage accounts).

That short function is 90% of Azure Policy's evaluation logic. The real service adds aliases, more operators (`like`, `contains`, `greater`), and parameters — but the shape is identical.

## 2. Initiatives — bundle policies and score compliance

An **initiative** (a.k.a. policy set) is just a list of policy definitions assigned together. Defender for Cloud's *Regulatory compliance* tab is built on initiatives like:

| Initiative                                    | What it checks                                 |
|-----------------------------------------------|------------------------------------------------|
| **Microsoft Cloud Security Benchmark (MCSB)** | Azure-specific best practices (default in DfC) |
| **CIS Microsoft Azure Foundations Benchmark** | CIS controls                                   |
| **NIST SP 800-53**                            | US federal                                     |
| **ISO 27001**                                 | International standard                         |
| **PCI DSS**                                   | Payment card industry                          |

Below we score a small subscription against a tiny "mini-MCSB" initiative.

In [2]:
# An initiative is just: a name + a list of policies.
deny_http = require_https  # reuse from cell above

require_tls12 = {
    "properties": {
        "displayName": "Storage accounts must enforce TLS 1.2",
        "policyRule": {
            "if": {
                "allOf": [
                    {"field": "type", "equals": "Microsoft.Storage/storageAccounts"},
                    {"field": "minimumTlsVersion", "notEquals": "TLS1_2"},
                ]
            },
            "then": {"effect": "audit"},
        },
    }
}

require_nsg = {
    "properties": {
        "displayName": "Subnets must have an NSG attached",
        "policyRule": {
            "if": {
                "allOf": [
                    {"field": "type", "equals": "Microsoft.Network/virtualNetworks/subnets"},
                    {"field": "networkSecurityGroup", "exists": False},
                ]
            },
            "then": {"effect": "audit"},
        },
    }
}

mini_mcsb = {
    "name": "mini-MCSB",
    "policies": [deny_http, require_tls12, require_nsg],
}

subscription = [
    {"name": "sa-prod",    "type": "Microsoft.Storage/storageAccounts",
     "supportsHttpsTrafficOnly": True,  "minimumTlsVersion": "TLS1_0"},
    {"name": "sa-legacy",  "type": "Microsoft.Storage/storageAccounts",
     "supportsHttpsTrafficOnly": False, "minimumTlsVersion": "TLS1_2"},
    {"name": "subnet-app", "type": "Microsoft.Network/virtualNetworks/subnets",
     "networkSecurityGroup": {"id": "/subscriptions/.../nsg-app"}},
    {"name": "subnet-db",  "type": "Microsoft.Network/virtualNetworks/subnets"},
]


def score_initiative(resources, initiative):
    total, compliant, findings = 0, 0, []
    for policy in initiative["policies"]:
        for r in resources:
            effect = evaluate_policy(r, policy)
            # Scope: only count resources the policy applies to
            if effect == "compliant" and policy["properties"]["policyRule"]["if"]["allOf"][0]["equals"] != r["type"]:
                continue  # policy did not target this resource type
            total += 1
            if effect == "compliant":
                compliant += 1
            else:
                findings.append((r["name"], policy["properties"]["displayName"], effect))
    pct = (compliant / total * 100) if total else 100.0
    return compliant, total, pct, findings


c, t, pct, findings = score_initiative(subscription, mini_mcsb)
print(f"=== {mini_mcsb['name']} compliance: {c}/{t} ({pct:.0f}%) ===\n")
for name, policy, effect in findings:
    icon = {"deny": "⛔", "audit": "⚠️"}.get(effect, "❔")
    print(f"  {icon} {name:12s} — {policy} → {effect}")


=== mini-MCSB compliance: 3/6 (50%) ===

  ⛔ sa-legacy    — Storage accounts must use HTTPS → deny
  ⚠️ sa-prod      — Storage accounts must enforce TLS 1.2 → audit
  ⚠️ subnet-db    — Subnets must have an NSG attached → audit


### What Defender for Cloud shows you

That same **compliant / total** ratio becomes the *compliance percentage* you see in Defender for Cloud's *Regulatory compliance* blade. Each row under a standard is a policy in the initiative; each "failed" control maps to non-compliant resources like the ones above.

## 3. Key Vault — access control decision helper

Azure Key Vault supports two permission models. You must pick **one per vault**:

| Axis                  | Vault access policy (legacy) | Azure RBAC (recommended)     |
|-----------------------|------------------------------|------------------------------|
| Granularity           | Per vault                    | Per key / secret / certificate |
| Inherit from parent?  | ❌                           | ✅ (RG / sub / MG)           |
| ABAC conditions       | ❌                           | ✅                           |
| Auditable in Entra?   | Limited                      | Full activity log            |

```bash
# Flip a vault to RBAC mode (existing policies are ignored once flipped)
az keyvault update -n my-kv -g rg-prod --enable-rbac-authorization true
```

In [3]:
# Decide which auth model to recommend for a Key Vault request.
def recommend_kv_access(scenario):
    if scenario.get("needs_per_secret_permissions"):
        return "✅ Use **Azure RBAC**. Per-secret data-plane roles give least privilege."
    if scenario.get("cross_subscription_apps"):
        return "✅ Use **Azure RBAC** and inherit role assignments from management group."
    if scenario.get("legacy_app_cant_change"):
        return "🟡 Keep the legacy **access policy** model (rare). Document why and revisit."
    return "✅ Default to **Azure RBAC** — it's the modern path and required for ABAC."


scenarios = [
    ("Modern web app, managed identity, only needs 'db-password'",
     {"needs_per_secret_permissions": True}),
    ("Old PowerShell script in production, legacy SDK, can't update",
     {"legacy_app_cant_change": True}),
    ("Greenfield multi-sub platform",
     {"cross_subscription_apps": True}),
]

for title, s in scenarios:
    print(f"• {title}\n  → {recommend_kv_access(s)}\n")


• Modern web app, managed identity, only needs 'db-password'
  → ✅ Use **Azure RBAC**. Per-secret data-plane roles give least privilege.

• Old PowerShell script in production, legacy SDK, can't update
  → 🟡 Keep the legacy **access policy** model (rare). Document why and revisit.

• Greenfield multi-sub platform
  → ✅ Use **Azure RBAC** and inherit role assignments from management group.



## 4. Soft delete & purge protection — the lifecycle

These two settings are the single biggest reason people pass or fail the "recover a deleted secret" exam questions.

* **Soft delete** (on by default, can't be turned off) keeps deleted objects for 7–90 days.
* **Purge protection** (opt-in, one-way) prevents *any* principal — even a Key Vault Administrator — from hard-deleting during the retention window.

```bash
az keyvault update -n my-kv --enable-purge-protection true   # cannot be undone
```

In [4]:
from datetime import datetime, timedelta

class Vault:
    """Mini Key Vault that enforces soft-delete + purge-protection semantics."""
    def __init__(self, name, retention_days=90, purge_protection=False):
        self.name = name
        self.retention_days = retention_days
        self.purge_protection = purge_protection
        self.active = {}           # name -> value
        self.soft_deleted = {}     # name -> (value, deleted_at)

    def set_secret(self, name, value):
        self.active[name] = value

    def delete_secret(self, name):
        if name in self.active:
            self.soft_deleted[name] = (self.active.pop(name), datetime.now())

    def recover(self, name):
        if name in self.soft_deleted:
            value, _ = self.soft_deleted.pop(name)
            self.active[name] = value
            return "recovered"
        return "not found"

    def purge(self, name, now=None):
        if name not in self.soft_deleted:
            return "not found"
        if self.purge_protection:
            _, deleted_at = self.soft_deleted[name]
            # Even the admin can't purge before retention is up
            deadline = deleted_at + timedelta(days=self.retention_days)
            if (now or datetime.now()) < deadline:
                return f"denied — purge protection blocks until {deadline:%Y-%m-%d}"
        del self.soft_deleted[name]
        return "purged"


kv = Vault("kv-prod", retention_days=90, purge_protection=True)
kv.set_secret("db-password", "s3cr3t!")
kv.delete_secret("db-password")
print("recover →", kv.recover("db-password"))    # ✅ works
kv.delete_secret("db-password")
print("purge now →", kv.purge("db-password"))    # ❌ blocked by purge protection
print("purge in 100d →", kv.purge("db-password", now=datetime.now() + timedelta(days=100)))


recover → recovered
purge now → denied — purge protection blocks until 2026-07-19
purge in 100d → purged


## 5. Key rotation — who's overdue?

Azure Key Vault can rotate keys automatically. A **rotation policy** is attached to a key and fires when:

* `timeAfterCreate` elapses → rotate.
* `timeBeforeExpiry` elapses → rotate (so the key never expires mid-flight).

```json
{
  "lifetimeActions": [
    {"trigger": {"timeAfterCreate": "P90D"}, "action": {"type": "Rotate"}},
    {"trigger": {"timeBeforeExpiry": "P30D"}, "action": {"type": "Notify"}}
  ],
  "attributes": {"expiryTime": "P1Y"}
}
```

Below we scan a fleet of keys and print which ones are due today.

In [5]:
from datetime import datetime, timedelta

today = datetime(2026, 4, 20)  # pin for reproducible output

keys = [
    {"name": "cmk-storage",  "created": datetime(2026, 1,  1), "expires": datetime(2026, 7,  1), "rotate_after_days": 90},
    {"name": "cmk-sql",      "created": datetime(2025, 10, 1), "expires": datetime(2026, 10, 1), "rotate_after_days": 90},
    {"name": "cmk-backups",  "created": datetime(2026, 3, 15), "expires": datetime(2027, 3, 15), "rotate_after_days": 365},
    {"name": "cmk-legacy",   "created": datetime(2024, 5,  1), "expires": datetime(2026, 5,  1), "rotate_after_days": 365},
]


def rotation_status(key, now):
    rotate_due = key["created"] + timedelta(days=key["rotate_after_days"])
    near_expiry = key["expires"] - timedelta(days=30)
    if now >= rotate_due and now >= near_expiry:
        return "🔴 rotate NOW (overdue + near expiry)"
    if now >= rotate_due:
        return "🟡 rotate (policy age reached)"
    if now >= near_expiry:
        return "🟡 rotate (expires within 30 days)"
    return "✅ ok"


print(f"=== Rotation scan on {today:%Y-%m-%d} ===")
for k in keys:
    print(f"  {k['name']:12s} created {k['created']:%Y-%m-%d}  expires {k['expires']:%Y-%m-%d}  → {rotation_status(k, today)}")


=== Rotation scan on 2026-04-20 ===
  cmk-storage  created 2026-01-01  expires 2026-07-01  → 🟡 rotate (policy age reached)
  cmk-sql      created 2025-10-01  expires 2026-10-01  → 🟡 rotate (policy age reached)
  cmk-backups  created 2026-03-15  expires 2027-03-15  → ✅ ok
  cmk-legacy   created 2024-05-01  expires 2026-05-01  → 🔴 rotate NOW (overdue + near expiry)


## 6. Bad → best: hardening a Key Vault

The same vault, progressively hardened. Copy this pattern into your own infra templates.

```bash
# ❌ BAD — defaults only. Public network, no purge protection, no diagnostics.
az keyvault create -g rg-prod -n kv-bad --location eastus

# 🟡 BETTER — RBAC + soft delete (on by default) + purge protection.
az keyvault create -g rg-prod -n kv-better --location eastus \
  --enable-rbac-authorization true \
  --enable-purge-protection true \
  --retention-days 90

# 🟢 BEST — private endpoint, public access OFF, diagnostics to Log Analytics.
az keyvault update -n kv-best --public-network-access Disabled
az network private-endpoint create -n pe-kv-best -g rg-prod \
  --vnet-name vnet-prod --subnet snet-pe \
  --private-connection-resource-id $(az keyvault show -n kv-best --query id -o tsv) \
  --group-id vault --connection-name kv-best
az monitor diagnostic-settings create -n kv-diag \
  --resource $(az keyvault show -n kv-best --query id -o tsv) \
  --workspace la-sentinel \
  --logs '[{"category":"AuditEvent","enabled":true},{"category":"AzurePolicyEvaluationDetails","enabled":true}]'
```

### Checklist you can drop into a PR review

In [6]:
KV_CHECKLIST = [
    ("RBAC authorization",      "az keyvault update --enable-rbac-authorization true",    "Per-secret least privilege + ABAC."),
    ("Soft delete",             "(on by default — cannot be disabled)",                   "Recover deleted items for N days."),
    ("Purge protection",        "az keyvault update --enable-purge-protection true",      "Admins can't hard-delete during retention."),
    ("Disable public access",   "az keyvault update --public-network-access Disabled",    "Force private endpoints only."),
    ("Private endpoint",        "az network private-endpoint create ... --group-id vault","Traffic never touches the Internet."),
    ("Diagnostics to LA",       "az monitor diagnostic-settings create ... --logs ...",   "Feeds Sentinel for audit/detection."),
    ("Key rotation policy",     "az keyvault key rotation-policy update ...",             "Auto-rotate every N days."),
    ("Backup/restore plan",     "az keyvault secret backup ...",                          "Same tenant + same geography only."),
]

print("=== Key Vault hardening checklist ===\n")
for setting, cli, why in KV_CHECKLIST:
    print(f"☐ {setting}")
    print(f"  why : {why}")
    print(f"  cli : {cli}\n")


=== Key Vault hardening checklist ===

☐ RBAC authorization
  why : Per-secret least privilege + ABAC.
  cli : az keyvault update --enable-rbac-authorization true

☐ Soft delete
  why : Recover deleted items for N days.
  cli : (on by default — cannot be disabled)

☐ Purge protection
  why : Admins can't hard-delete during retention.
  cli : az keyvault update --enable-purge-protection true

☐ Disable public access
  why : Force private endpoints only.
  cli : az keyvault update --public-network-access Disabled

☐ Private endpoint
  why : Traffic never touches the Internet.
  cli : az network private-endpoint create ... --group-id vault

☐ Diagnostics to LA
  why : Feeds Sentinel for audit/detection.
  cli : az monitor diagnostic-settings create ... --logs ...

☐ Key rotation policy
  why : Auto-rotate every N days.
  cli : az keyvault key rotation-policy update ...

☐ Backup/restore plan
  why : Same tenant + same geography only.
  cli : az keyvault secret backup ...



## Key points to remember

| Concept                | What matters on the exam                                                                    |
|------------------------|---------------------------------------------------------------------------------------------|
| Policy effects         | Know the 6: Audit, Deny, Modify, DeployIfNotExists, AuditIfNotExists, Disabled.             |
| Remediation            | Modify / DeployIfNotExists need a managed identity + a *remediation task* for old resources. |
| Initiatives            | MCSB is the default in Defender for Cloud. Custom initiatives power custom standards.        |
| Key Vault auth         | Prefer RBAC. Legacy access policy is per-vault only.                                        |
| Soft delete            | Always on. Purge protection is optional and **one-way**.                                    |
| Key rotation           | Automated with a rotation policy. `timeAfterCreate` and `timeBeforeExpiry`.                 |
| Backup/restore         | Same tenant + same Azure geography. Not cross-tenant.                                       |

**Next**: [Notebook 2 — Defender for Cloud](02_defender_for_cloud.ipynb)